# DataSense Hackathon — Final Decision Test Harness

**Purpose:** run a fast, repeatable benchmark to decide which real-world problem statement to build in 2 days.

This notebook does **not** try to prove that the LLM itself is the bottleneck. Your profiling already showed that the current DataSense loop is mostly **model generation latency**, while GPU acceleration matters most on certain *data operations* and *ML training* workloads.  
The job here is to identify the most defensible hackathon problem statement by testing which workload shows the strongest GPU win at scale.

**Candidate directions this notebook helps compare**
- **Risk / priority scoring** → classification / triage
- **Time-series alerting** → rolling metrics / freshness
- **Dashboard backend** → groupby / join / aggregation
- **Forecasting** → regression

**What to look for**
- large and consistent speedup
- the speedup survives at bigger row counts
- the workload maps cleanly to a real user and a useful decision


## 0. Install RAPIDS

Kaggle T4 runtime, CUDA 12. This is usually the slowest cell. Run it first.


In [ ]:
%%capture
!pip install -q --extra-index-url=https://pypi.nvidia.com \
    cudf-cu12 cuml-cu12 cupy-cuda12x
!pip install -q scikit-learn matplotlib pandas


In [ ]:
import time
import numpy as np
import pandas as pd

# Optional imports: keep the notebook readable even if a GPU package import fails.
try:
    import cudf
    import cuml
    RAPIDS_OK = True
except Exception as e:
    RAPIDS_OK = False
    cudf = None
    cuml = None
    print("RAPIDS import failed:", e)

print("RAPIDS available:", RAPIDS_OK)
if RAPIDS_OK:
    print("cuDF:", cudf.__version__, "| cuML:", cuml.__version__)
    !nvidia-smi --query-gpu=name,memory.total --format=csv


## 1. Synthetic data generator

The data schema is intentionally generic so the result can map to retail, support, finance, logistics, or ops.

You can change `SIZES` based on how much time you have:
- quick test: `[100_000, 1_000_000]`
- balanced test: `[100_000, 1_000_000, 5_000_000]`
- full test: `[100_000, 1_000_000, 5_000_000, 20_000_000]`


In [ ]:
SIZES = [100_000, 1_000_000, 5_000_000, 20_000_000]
N_STORES, N_PRODUCTS, N_USERS, N_REGIONS = 200, 2000, 50_000, 8

def make_transactions(n, seed=42):
    rng = np.random.default_rng(seed)
    return pd.DataFrame({
        "store_id": rng.integers(0, N_STORES, n),
        "product_id": rng.integers(0, N_PRODUCTS, n),
        "user_id": rng.integers(0, N_USERS, n),
        "region": rng.integers(0, N_REGIONS, n),
        "timestamp": pd.to_datetime("2024-01-01") + pd.to_timedelta(
            rng.integers(0, 365 * 24 * 3600, n), unit="s"
        ),
        "amount": rng.exponential(40, n).round(2),
        "qty": rng.integers(1, 10, n),
        "discount_pct": rng.uniform(0, 0.3, n).round(3),
        "is_return": rng.random(n) < 0.05,
        "feat_1": rng.normal(0, 1, n),
        "feat_2": rng.normal(0, 1, n),
        "feat_3": rng.normal(0, 1, n),
        "feat_4": rng.normal(0, 1, n),
    })

def add_target(df):
    # weakly correlated target for classification / scoring
    score = (
        df["feat_1"] * 0.6
        + df["feat_2"] * -0.4
        + df["amount"] / 100
        + np.random.default_rng(0).normal(0, 1, len(df))
    )
    df["target_flag"] = (score > score.median()).astype(int)
    df["target_value"] = score
    return df

print("Example preview:")
display(add_target(make_transactions(5)).head())


## 2. Timing harness

In [ ]:
results = []  # rows: {op, size, engine, seconds}

def timeit(fn, *args, **kwargs):
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return out, time.perf_counter() - t0

def record(op, size, engine, seconds):
    results.append({"op": op, "size": size, "engine": engine, "seconds": seconds})
    print(f"  [{engine:6s}] {op:22s} n={size:>11,} -> {seconds:8.3f}s")


## 3. Data-wrangling sweep: groupby / join / sort / rolling-window

This is the signal for a **dashboard backend** or **time-series alerting** style project.


In [ ]:
if not RAPIDS_OK:
    raise RuntimeError("RAPIDS is required for this benchmark notebook.")

for n in SIZES:
    print(f"\n=== n={n:,} ===")
    pdf = add_target(make_transactions(n))
    gdf = cudf.DataFrame.from_pandas(pdf)

    # groupby + agg
    _, t = timeit(lambda: pdf.groupby(["store_id", "product_id"]).agg({"amount": "sum", "qty": "sum"}))
    record("groupby_agg", n, "pandas", t)
    _, t = timeit(lambda: gdf.groupby(["store_id", "product_id"]).agg({"amount": "sum", "qty": "sum"}))
    record("groupby_agg", n, "cudf", t)

    # join
    store_dim_pd = pd.DataFrame({
        "store_id": range(N_STORES),
        "store_tier": np.random.choice(["A", "B", "C"], N_STORES)
    })
    store_dim_gd = cudf.DataFrame.from_pandas(store_dim_pd)

    _, t = timeit(lambda: pdf.merge(store_dim_pd, on="store_id", how="left"))
    record("merge_join", n, "pandas", t)
    _, t = timeit(lambda: gdf.merge(store_dim_gd, on="store_id", how="left"))
    record("merge_join", n, "cudf", t)

    # sort
    _, t = timeit(lambda: pdf.sort_values("amount"))
    record("sort", n, "pandas", t)
    _, t = timeit(lambda: gdf.sort_values("amount"))
    record("sort", n, "cudf", t)

    # rolling window
    _, t = timeit(lambda: pdf.sort_values("timestamp").groupby("store_id")["amount"].rolling(7).mean())
    record("rolling_window", n, "pandas", t)
    try:
        _, t = timeit(lambda: gdf.sort_values("timestamp").groupby("store_id")["amount"].rolling(7).mean())
        record("rolling_window", n, "cudf", t)
    except Exception as e:
        print("  [cudf] rolling_window unsupported/errored:", e)


## 4. ML sweep: forecasting / segmentation / risk scoring

This is the signal for a **forecasting** or **risk / priority scoring** style project.


In [ ]:
from sklearn.linear_model import LinearRegression as skLinearRegression
from sklearn.cluster import KMeans as skKMeans
from sklearn.ensemble import RandomForestClassifier as skRandomForestClassifier
from cuml.linear_model import LinearRegression as cuLinearRegression
from cuml.cluster import KMeans as cuKMeans
from cuml.ensemble import RandomForestClassifier as cuRandomForestClassifier

FEATURES = ["feat_1", "feat_2", "feat_3", "feat_4", "amount", "qty"]

for n in SIZES:
    print(f"\n=== n={n:,} ===")
    pdf = add_target(make_transactions(n))
    gdf = cudf.DataFrame.from_pandas(pdf)

    X_pd, y_reg_pd, y_clf_pd = pdf[FEATURES], pdf["target_value"], pdf["target_flag"]
    X_gd = gdf[FEATURES].astype("float32")
    y_reg_gd = gdf["target_value"].astype("float32")
    y_clf_gd = gdf["target_flag"].astype("int32")

    # regression / forecasting
    _, t = timeit(lambda: skLinearRegression().fit(X_pd, y_reg_pd))
    record("linreg_fit", n, "sklearn", t)
    _, t = timeit(lambda: cuLinearRegression().fit(X_gd, y_reg_gd))
    record("linreg_fit", n, "cuml", t)

    # clustering / segmentation
    _, t = timeit(lambda: skKMeans(n_clusters=8, n_init=3, random_state=0).fit(X_pd))
    record("kmeans_fit", n, "sklearn", t)
    _, t = timeit(lambda: cuKMeans(n_clusters=8, n_init=3, random_state=0).fit(X_gd))
    record("kmeans_fit", n, "cuml", t)

    # risk scoring / triage
    if n <= 5_000_000:
        _, t = timeit(lambda: skRandomForestClassifier(n_estimators=50, max_depth=10, n_jobs=-1).fit(X_pd, y_clf_pd))
        record("rf_classify_fit", n, "sklearn", t)
    else:
        print(f"  [sklearn] rf_classify_fit skipped at n={n:,} (too slow on CPU)")
    _, t = timeit(lambda: cuRandomForestClassifier(n_estimators=50, max_depth=10).fit(X_gd, y_clf_gd))
    record("rf_classify_fit", n, "cuml", t)


## 5. Results table and speedup ranking

In [ ]:
res_df = pd.DataFrame(results)
res_df.to_csv("gpu_sweep_results.csv", index=False)

# Build CPU/GPU speedup rows
speedups = []
for op in res_df["op"].unique():
    sub = res_df[res_df["op"] == op]
    cpu_engine = "pandas" if "pandas" in sub["engine"].values else "sklearn"
    gpu_engine = "cudf" if "cudf" in sub["engine"].values else "cuml"
    for n in sorted(sub["size"].unique()):
        cpu_t = sub[(sub["size"] == n) & (sub["engine"] == cpu_engine)]["seconds"]
        gpu_t = sub[(sub["size"] == n) & (sub["engine"] == gpu_engine)]["seconds"]
        if len(cpu_t) and len(gpu_t) and gpu_t.values[0] > 0:
            speedups.append({
                "op": op,
                "size": n,
                "speedup_x": cpu_t.values[0] / gpu_t.values[0],
                "cpu_engine": cpu_engine,
                "gpu_engine": gpu_engine,
            })

speedup_df = pd.DataFrame(speedups).sort_values(["op", "size"])
print("Speedup (CPU seconds / GPU seconds):")
display(speedup_df.pivot(index="size", columns="op", values="speedup_x").round(1))

print("\nTop speedups overall:")
display(speedup_df.sort_values("speedup_x", ascending=False).head(12))


## 6. Candidate project fit score

This is the part that turns benchmark numbers into a hackathon choice.

The heuristic is simple:
- **high max speedup** is good
- **speedup that survives at 1M–20M rows** is better
- **the operation must map to a real user decision**


In [ ]:
import matplotlib.pyplot as plt

# summarize per-op
summary_rows = []
for op in speedup_df["op"].unique():
    sub = speedup_df[speedup_df["op"] == op].sort_values("size")
    max_speedup = sub["speedup_x"].max()
    min_speedup = sub["speedup_x"].min()
    mean_speedup = sub["speedup_x"].mean()
    best_size = int(sub.loc[sub["speedup_x"].idxmax(), "size"])
    summary_rows.append({
        "op": op,
        "min_speedup": min_speedup,
        "mean_speedup": mean_speedup,
        "max_speedup": max_speedup,
        "best_size": best_size,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("max_speedup", ascending=False)
display(summary_df.round(2))

# map ops to idea statements
idea_map = {
    "rf_classify_fit": "Risk / priority scoring (ticket triage, fraud, ops priority)",
    "rolling_window": "Time-series alerting (freshness, anomaly tracking, rolling metrics)",
    "groupby_agg": "Dashboard backend / BI acceleration",
    "merge_join": "Dashboard backend / enrichment joins",
    "linreg_fit": "Forecasting dashboard (demand, revenue, inventory)",
    "kmeans_fit": "Segmentation (weakest option here)",
}

def idea_label(op):
    return idea_map.get(op, op)

summary_df["project_direction"] = summary_df["op"].map(idea_label)
display(summary_df[["project_direction", "max_speedup", "mean_speedup", "min_speedup", "best_size"]].round(2))

print("\nRecommended ranking:")
for i, row in enumerate(summary_df.itertuples(index=False), start=1):
    print(f"{i}. {idea_label(row.op)}  |  max={row.max_speedup:.1f}x  mean={row.mean_speedup:.1f}x  best_size={row.best_size:,}")


## 7. Visualize the speedups

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for op in speedup_df["op"].unique():
    sub = speedup_df[speedup_df["op"] == op].sort_values("size")
    ax.plot(sub["size"], sub["speedup_x"], marker="o", label=op)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("rows")
ax.set_ylabel("speedup (CPU time / GPU time)")
ax.set_title("Where does GPU acceleration pay off?")
ax.axhline(1, color="gray", linestyle="--", linewidth=1)
ax.legend()
plt.tight_layout()
plt.savefig("gpu_speedup_sweep.png", dpi=150)
plt.show()


## 8. Final interpretation for the hackathon

Use the output of this notebook to choose the final project by this rule:

- **If `rf_classify_fit` wins clearly** → build a **risk / priority scoring** tool.
- **If `rolling_window` wins clearly** → build a **time-series alerting** tool.
- **If `groupby_agg` / `merge_join` wins clearly** → build a **dashboard backend / BI acceleration** tool.
- **If nothing is consistently >3–5×** → do not force the NVIDIA story; use a Google Cloud + Gemini decision-support angle instead.

This notebook is meant to be run **before** locking the final hackathon problem statement.


In [ ]:
# Optional: save a compact text summary for later
with open("final_decision_notes.txt", "w") as f:
    f.write("DataSense hackathon final test harness\n")
    f.write("Top speedups:\n")
    f.write(summary_df.to_string(index=False))
print("Saved gpu_sweep_results.csv, gpu_speedup_sweep.png, final_decision_notes.txt")
